# Speckle2Self: ultrasound speckle reduction

[![arXiv](https://img.shields.io/badge/arXiv-Paper-b31b1b.svg)](https://arxiv.org/abs/2507.06828) &nbsp; [![GitHub](https://img.shields.io/badge/GitHub-Code-black?logo=github)](https://github.com/noseefood/speckle2self)

This notebook demonstrates how to perform **self-supervised ultrasound speckle reduction** using [Speckle2Self](https://arxiv.org/abs/2507.06828) within the [zea](https://github.com/tue-bmd/zea) framework. We apply the model to an **in-vivo carotid artery** scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023), matching the domain on which the `speckle2self-invivo` weights were trained.

For more information on the method, see the [original paper](https://arxiv.org/abs/2507.06828):
- Lin, Huiping, et al. "Speckle2Self: Learning Self-Supervised Despeckling with Attention Mechanism for SAR Images." Remote Sensing 17.23 (2025): 3840.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb) &nbsp; [![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/models/speckle2self_despeckling_example.ipynb)
[![Hugging Face model](https://img.shields.io/badge/Hugging%20Face-Model-yellow?logo=huggingface)](https://huggingface.co/zeahub/speckle2self-invivo)

‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [1]:
%%capture
%pip install zea

In [2]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
import matplotlib.pyplot as plt

import zea
from zea.ops import (
    Beamform,
    EnvelopeDetect,
    Pipeline,
    Demodulate,
    Downsample,
    Normalize,
    LogCompress,
)
from zea.visualize import set_mpl_style

zea: Using backend 'tensorflow'


In [3]:
zea.init_device(verbose=False)
set_mpl_style()

In [4]:
carotid_path = "hf://zeahub/picmus/in_vivo/carotid_cross/carotid_cross_expe_dataset_rf/carotid_cross_expe_dataset_rf.hdf5"
frame_idx = 0
n_tx = 11  # transmits per frame (fewer -> faster; more -> better quality)
dynamic_range = (-40, 0)  # dB

## Load carotid data

We load a single frame of an in-vivo carotid scan from [zeahub/zea-carotid-2023](https://huggingface.co/datasets/zeahub/zea-carotid-2023) on Hugging Face. Note that this is raw ultrasound RF channel data. Therefore we will first need to beamform and envelope-detect the data before feeding it to the Speckle2Self model. We will do this using the `zea.Pipeline` framework, which provides efficient implementations of these operations. For more details see the [pipeline example notebook](../pipeline/zea_pipeline_example.ipynb) or the [API reference](https://zea.readthedocs.io/en/latest/pipeline.html).

In [5]:
with zea.File(carotid_path, revision="v0.1.0a1") as f:
    data = f.data.raw_data[frame_idx][None, ...]
    scan = f.scan
    probe = f.probe

scan.set_transmits(n_tx)
scan.zlims = (0, 0.04)
scan.grid_size_x = 500
scan.grid_size_z = 800

pipeline = Pipeline(
    operations=[
        Demodulate(),
        Downsample(factor=2),
        Beamform(
            beamformer="delay_and_sum",
            enable_pfield=True,
            num_patches=100,
        ),
        EnvelopeDetect(),
        Normalize(),
        LogCompress(),
    ],
    with_batch_dim=False,
    jit_options="pipeline",
)
parameters = pipeline.prepare_parameters(probe, scan, dynamic_range=dynamic_range)

zea: WARNING This ``zea.File`` '/root/.cache/zea/huggingface/datasets/datasets--zeahub--picmus/snapshots/3130fc6f78f6613de5b23da9233a8a130a89c830/in_vivo/carotid_cross/carotid_cross_expe_dataset_rf/carotid_cross_expe_dataset_rf.hdf5' was created with a legacy version of zea (0.1.0a0), while you are using zea v0.1.0a0. It may behave in unexpected ways. Install an earlier version of zea<0.1.0 for full compatibility or re-save the file with zea v0.1.0 or later (e.g. via File.create).
zea: Loading cached result for compute_pfield.
zea: WARNING No ``tx_waveform_indices`` provided, using zeros


In [6]:
# Beamform + envelope-detect
out = pipeline(data=data[0, scan.selected_transmits], **parameters)
envelope = out[pipeline.output_key]
envelope = keras.ops.convert_to_numpy(envelope)

zea: DEBUG [zea.Pipeline] The following input keys are not used by the pipeline: {'center_frequency', 'element_height', 'element_width', 'bandwidth_percent'}. Make sure this is intended. This warning will only be shown once.


## Load Speckle2Self model

`Speckle2Self` can be loaded directly from Hugging Face:

In [7]:
from zea.models.speckle2self import Speckle2Self

model = Speckle2Self.from_preset("hf://zeahub/speckle2self-invivo")

## Preprocess and run inference

In [8]:
# Run despeckling (Speckle2Self) model inference
despeckled = model(envelope[None, ..., None])
despeckled = keras.ops.convert_to_numpy(despeckled)
despeckled = despeckled.squeeze()

## Visualize results

For a simple comparison, we display one frame before and after Speckle2Self despeckling.


In [9]:
xlims_mm = [v * 1e3 for v in scan.xlims]
zlims_mm = [v * 1e3 for v in scan.zlims]
extent = [xlims_mm[0], xlims_mm[1], zlims_mm[1], zlims_mm[0]]

# gamma correction for better visualization
despeckled_viz = np.power(despeckled, 1.3)

fig, axes = plt.subplots(1, 2, figsize=(8, 4), squeeze=False)

axes[0, 0].imshow(envelope, cmap="gray", extent=extent)
axes[0, 0].set_title("B-Mode")
axes[0, 0].set_xlabel("X (mm)")
axes[0, 0].set_ylabel("Z (mm)")

axes[0, 1].imshow(despeckled_viz, cmap="gray", extent=extent)
axes[0, 1].set_title("Despeckled B-Mode (Speckle2Self)")
axes[0, 1].set_xlabel("X (mm)")
axes[0, 1].set_ylabel("Z (mm)")

plt.tight_layout()
plt.savefig("speckle2self_output.png", bbox_inches="tight", dpi=100)
plt.close()

**Speckle2Self result on in-vivo carotid data**

![Image](./speckle2self_output.png)